In [2]:
# ============================================================
# ReAct RAG Agent — LangGraph + LangSmith + LangChain (modern)
# ============================================================
# pip install -U langchain langchain-groq langchain-community
#             langgraph langsmith tavily-python
#             arxiv wikipedia python-dotenv rich

from __future__ import annotations

import os
from typing import Annotated, Literal
from dotenv import load_dotenv
from rich import print
from rich.panel import Panel

load_dotenv()

# ── LangSmith tracing (set vars before importing langchain) ──────────────────
os.environ["LANGCHAIN_TRACING_V2"]  = "true"
os.environ["LANGCHAIN_PROJECT"]     = "react-rag-demo"          # project name in LangSmith UI
os.environ["LANGCHAIN_API_KEY"]     = os.getenv("LANGCHAIN_API_KEY", "")
os.environ["GROQ_API_KEY"]          = os.getenv("GROQ_API_KEY", "")
os.environ["TAVILY_API_KEY"]        = os.getenv("TAVILY_API_KEY", "")

# ── LangGraph / LangChain imports ────────────────────────────────────────────
from langchain_groq import ChatGroq
from langchain_core.messages import HumanMessage, AIMessage, BaseMessage, ToolMessage
from langchain_core.tools import tool

from langchain_community.tools import ArxivQueryRun, WikipediaQueryRun
from langchain_community.utilities import ArxivAPIWrapper, WikipediaAPIWrapper
from langchain_community.tools.tavily_search import TavilySearchResults

from langgraph.graph import StateGraph, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode, tools_condition
from langgraph.checkpoint.memory import MemorySaver

from typing_extensions import TypedDict


# ════════════════════════════════════════════════════════════
# 1. LLM
# ════════════════════════════════════════════════════════════
llm = ChatGroq(
    model="meta-llama/llama-4-scout-17b-16e-instruct",
    temperature=0,
    streaming=True,         # token-by-token via .stream()
)


# ════════════════════════════════════════════════════════════
# 2. Tools  (wrapped with @tool for cleaner docstrings)
# ════════════════════════════════════════════════════════════
@tool
def search_arxiv(query: str) -> str:
    """Search academic papers on arXiv. Best for ML, physics, math, CS research."""
    wrapper = ArxivAPIWrapper(top_k_results=2, doc_content_chars_max=600)
    runner  = ArxivQueryRun(api_wrapper=wrapper)
    return runner.invoke(query)


@tool
def search_wikipedia(query: str) -> str:
    """Search Wikipedia for factual background knowledge and definitions."""
    wrapper = WikipediaAPIWrapper(top_k_results=1, doc_content_chars_max=600)
    runner  = WikipediaQueryRun(api_wrapper=wrapper)
    return runner.invoke(query)


@tool
def search_web(query: str) -> str:
    """Search the live web for current news, events, and up-to-date information."""
    return TavilySearchResults(max_results=3).invoke(query)


tools = [search_arxiv, search_wikipedia, search_web]

# Bind tools to the LLM so it can emit tool-call messages natively
llm_with_tools = llm.bind_tools(tools)


# ════════════════════════════════════════════════════════════
# 3. State — the shared object that flows through the graph
# ════════════════════════════════════════════════════════════
class AgentState(TypedDict):
    # add_messages reducer: new messages are appended, not replaced
    messages: Annotated[list[BaseMessage], add_messages]


# ════════════════════════════════════════════════════════════
# 4. Graph nodes
# ════════════════════════════════════════════════════════════
def call_llm(state: AgentState) -> dict:
    """LLM node — reasons and optionally emits a tool-call."""
    response = llm_with_tools.invoke(state["messages"])
    return {"messages": [response]}


# ToolNode automatically:
# • finds the right @tool function from the tool-call message
# • runs it and wraps the result in a ToolMessage
tool_node = ToolNode(tools)


# ════════════════════════════════════════════════════════════
# 5. Build the graph
# ════════════════════════════════════════════════════════════
#
#   ┌──────────┐    tool call?    ┌───────────┐
#   │  call_llm │ ──── yes ──────▶│ tool_node │
#   │          │◀──── loop ───────│           │
#   └──────────┘                  └───────────┘
#         │
#      no tool call
#         ▼
#        END
#
builder = StateGraph(AgentState)

builder.add_node("llm",   call_llm)
builder.add_node("tools", tool_node)

builder.set_entry_point("llm")

# tools_condition returns "tools" if the LLM emitted a tool call, else END
builder.add_conditional_edges("llm", tools_condition)
builder.add_edge("tools", "llm")   # after tool runs → back to LLM


# ── Memory checkpointer (persists conversation across .invoke() calls) ───────
memory = MemorySaver()
graph  = builder.compile(checkpointer=memory)


# ════════════════════════════════════════════════════════════
# 6. Helper: run a query with streaming output
# ════════════════════════════════════════════════════════════
def run(query: str, thread_id: str = "default") -> str:
    """
    Run the agent for a given query.
    thread_id lets you maintain separate conversation histories.
    """
    config = {"configurable": {"thread_id": thread_id}}
    inputs = {"messages": [HumanMessage(content=query)]}

    print(Panel(f"[bold cyan]Query:[/bold cyan] {query}", expand=False))

    final_answer = ""

    # stream_mode="updates" yields {node_name: state_delta} dicts
    for update in graph.stream(inputs, config=config, stream_mode="updates"):
        for node_name, state_delta in update.items():
            for msg in state_delta.get("messages", []):
                if isinstance(msg, AIMessage):
                    if msg.tool_calls:
                        for tc in msg.tool_calls:
                            print(f"  [yellow]→ Tool call:[/yellow] [bold]{tc['name']}[/bold]({tc['args']})")
                    elif msg.content:
                        print(f"  [green]← LLM:[/green] {msg.content}")
                        final_answer = msg.content
                elif isinstance(msg, ToolMessage):
                    snippet = str(msg.content)[:200].replace("\n", " ")
                    print(f"  [blue]  Observation:[/blue] {snippet}…")

    print()
    return final_answer


# ════════════════════════════════════════════════════════════
# 7. Multi-turn conversation example (shared thread memory)
# ════════════════════════════════════════════════════════════
def run_conversation():
    thread = "session-1"

    turns = [
        "What is Retrieval Augmented Generation (RAG)?",
        "What are the most recent papers about RAG on arXiv?",
        "What are today's top AI news headlines?",
        "Summarise everything you've told me so far.",   # uses in-memory history
    ]

    for query in turns:
        run(query, thread_id=thread)
        print("─" * 60)


# ════════════════════════════════════════════════════════════
# 8. Visualise the graph (optional, writes graph.png)
# ════════════════════════════════════════════════════════════
def save_graph_image():
    try:
        img_bytes = graph.get_graph().draw_mermaid_png()
        with open("graph.png", "wb") as f:
            f.write(img_bytes)
        print("[green]Graph saved to graph.png[/green]")
    except Exception as e:
        print(f"[yellow]Could not save graph image: {e}[/yellow]")


# ════════════════════════════════════════════════════════════
# 9. Entry point
# ════════════════════════════════════════════════════════════
if __name__ == "__main__":
    run_conversation()
    save_graph_image()

╭──────────────────────────────────────────────────────╮
│ Query: What is Retrieval Augmented Generation (RAG)? │
╰──────────────────────────────────────────────────────╯

→ Tool call: search_wikipedia({'query': 'Retrieval Augmented Generation'})

  Observation: Page: Retrieval-augmented generation Summary: Retrieval-augmented generation (RAG) is a technique 
that enables large language models (LLMs) to retrieve and incorporate new information from external da…

← LLM: Retrieval-augmented generation (RAG) is a technique that enables large language models (LLMs) to retrieve 
and incorporate new information from external data sources. With RAG, LLMs first refer to a specified set of 
documents, then respond to user queries. These documents supplement information from the LLM's pre-existing 
training data. This allows LLMs to use domain-specific and/or updated information that is not available in the 
training data. For example, this helps LLM-based chatbots access internal company data or generate responses based 
on available data.

────────────────────────────────────────────────────────────

╭────────────────────────────────────────────────────────────╮
│ Query: What are the most recent papers about RAG on arXiv? │
╰────────────────────────────────────────────────────────────╯

→ Tool call: search_arxiv({'query': 'Retrieval Augmented Generation RAG'})

AttributeError: 'Search' object has no attribute 'results'